In [344]:
import os
import random
from utils.hand_model_lite import HandModelMJCFLite
import numpy as np
import transforms3d
import torch
import trimesh


In [345]:
# import numpy as np

# # Заменить 'имя_файла.npy' на путь к твоему файлу
# data = np.load('../data/dataset/hummer.npy', allow_pickle=True)

# print(data)

In [346]:
mesh_path = "../data/meshdata"
# hand_name ="shadow_dexee"
# hand_name ="barret"
hand_name ="DIP-Flex_opened_kinematics"
# hand_name = "robotiq_2"
# hand_name = "panda"

use_visual_mesh = True

if hand_name =="shadow_dexee":
    '''For shadow dexee'''
    data_path = "../data/dataset/shadow_dexee/"
    hand_file = "mjcf/shadow_dexee.xml"
    joint_names = [
                    "F0_J0", "F0_J1", "F0_J2", "F0_J3", "F1_J0", "F1_J1", "F1_J2", "F1_J3", "F2_J0", "F2_J1", "F2_J2", "F2_J3"
    ]

elif hand_name =="barret":
    ''' For BarretHand'''
    data_path = "../data/dataset/barret/"
    hand_file = "mjcf/barret.xml"
    joint_names = [
                    "wam_bhand_finger_1_prox_joint", "wam_bhand_finger_1_med_joint", "wam_bhand_finger_1_dist_joint", 
                    "wam_bhand_finger_2_prox_joint", "wam_bhand_finger_2_med_joint", "wam_bhand_finger_2_dist_joint",
                    "wam_bhand_finger_3_med_joint", "wam_bhand_finger_3_dist_joint"
    ]

elif hand_name =="DIP-Flex_opened_kinematics":
    ''' For Egorhand'''
    data_path = "../data/dataset/DIP-Flex_opened_kinematics"
    hand_file = "mjcf/DIP-Flex_opened_kinematics.xml"
    joint_names = [
                    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                    "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                    "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
    ]

elif hand_name == "robotiq_2":
    '''For robotiq'''
    data_path = "../data/dataset/robotiq_2/"
    hand_file = "mjcf/robotiq_2 simpl.xml"
    joint_names = [
                    "left_spring_link_joint", "left_follower",
                    "right_spring_link_joint", "right_follower_joint"
    ]

elif hand_name == "panda":
    '''For panda'''
    data_path = "../data/dataset/panda/"
    hand_file = "mjcf/panda.xml"
    joint_names = [
                    "finger_joint1", "finger_joint2"
    ]

translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']


In [347]:
hand_model = HandModelMJCFLite(
    hand_file,
    "mjcf/assets/" + hand_name)

In [348]:
grasp_code_list = []
for code in os.listdir(data_path):
    grasp_code_list.append(code[:-4])

print(grasp_code_list)

['core-mug-8570d9a8d24cb0acbebd3c0c0c70fb03', 'pliers', 'screwdriver', 'hummer', 'ddg-gd_banana_poisson_002', 'sem-Camera-7bff4fd4dc53de7496dece3f86cb5dd5', 'sem-Bottle-437678d4bc6be981c8724d5673a063a6', 'mujoco-Ecoforms_Plant_Plate_S11Turquoise']


In [372]:
grasp_code = random.choice(grasp_code_list)
grasp_data = np.load(
    os.path.join(data_path, grasp_code+".npy"), allow_pickle=True)
object_mesh_origin = trimesh.load(os.path.join(
    mesh_path, grasp_code, "coacd/decomposed.obj"))
print(grasp_code)

print(grasp_data)
print(len(grasp_data))

sem-Camera-7bff4fd4dc53de7496dece3f86cb5dd5
[{'scale': 0.10000000149011612, 'qpos': {'Joint_pinkie_abduction': 0.07644443213939667, 'Joint_pinkie_PPflexion': 0.27078160643577576, 'Joint_pinkie_DPflexion': 0.49089500308036804, 'Joint_index_abduction': 0.3994938135147095, 'Joint_index_PPflexion': 0.3959054946899414, 'Joint_index_DPflexion': 0.2853049337863922, 'Joint_thumb_rotation': 0.1491260975599289, 'Joint_thumb_abduction': 1.2701289653778076, 'Joint_thumb_PPflexion': 0.039835888892412186, 'Joint_thumb_DPflexion': 0.09672065824270248, 'WRJRx': -2.5286838251756776, 'WRJRy': 1.0110157937372637, 'WRJRz': -0.10458244062796719, 'WRJTx': 0.04408043250441551, 'WRJTy': -0.17330335080623627, 'WRJTz': 0.07521923631429672}, 'qpos_st': {'Joint_pinkie_abduction': 0.029954900965094566, 'Joint_pinkie_PPflexion': 0.30032697319984436, 'Joint_pinkie_DPflexion': 0.18628144264221191, 'Joint_index_abduction': 0.4055754840373993, 'Joint_index_PPflexion': 0.21236856281757355, 'Joint_index_DPflexion': 0.278

In [390]:
index = random.randint(0, len(grasp_data) - 1)
# index = 3

qpos = grasp_data[index]['qpos']
print(index)
rot = np.array(transforms3d.euler.euler2mat(
    *[qpos[name] for name in rot_names]))
rot = rot[:, :2].T.ravel().tolist()
hand_pose = torch.tensor([qpos[name] for name in translation_names] + rot + [qpos[name]
                         for name in joint_names], dtype=torch.float, device="cpu").unsqueeze(0)
hand_model.set_parameters(hand_pose)
hand_mesh = hand_model.get_trimesh_data(0)
object_mesh = object_mesh_origin.copy().apply_scale(grasp_data[index]["scale"])

# Задаем цвета (RGB в формате [R, G, B, A], где значения от 0.0 до 1.0)
hand_color = [0.7, 0.7, 0.7, 1.0]  # Красноватый цвет для руки
object_color = [0.2, 0.5, 0.8, 1.0]  # Голубоватый цвет для объекта

# Применяем цвета к мешам
hand_mesh.visual.face_colors = hand_color
object_mesh.visual.face_colors = object_color


1


In [391]:
(hand_mesh+object_mesh).show()
